# Find companies and their filings in each market

Each market uses its own identifier. When you only know a name, search the
route first and pass the result straight to `Company`.

## Identify your SEC requests

The SEC asks automated clients for a name or organization and a contact email,
separated by a space. For example, `Alex Morgan alex@example.com` or
`Example Research filings@example.com`. These are examples: use your own name
and a working email. No SEC account or API key is needed.

In the next cell, uncomment the `os.environ[...]` line and replace the example
with your details. If you already set this environment variable in your
terminal, leave the line commented. The label is sent to the SEC with each
request. See the [setup guide](../../docs/getting-started.md#sec-request-identity).

In [1]:
import os
from collections import Counter

from opencorpfilings import Company, provider_info, search_companies

# Uncomment this line and replace the example with your own name and contact email.
# os.environ["OPENCORPFILINGS_SEC_USER_AGENT"] = "Alex Morgan alex@example.com"
assert os.getenv("OPENCORPFILINGS_SEC_USER_AGENT"), "Uncomment the SEC identity example above and replace it with your own details, or set OPENCORPFILINGS_SEC_USER_AGENT in your environment."

Not sure which market a company files in? `find_company` searches all of them by
name, ticker, securities code, ISIN or LEI. The first call builds a local index
from official lists (about 20 seconds); later calls use the cached copy.

In [2]:
from opencorpfilings import find_company

find_company("Toyota", limit=4)

[<CompanyMatch US 1094517 'TOYOTA MOTOR CORP/' supported>, <CompanyMatch JP 7203 'TOYOTA MOTOR CORPORATION' supported>, <CompanyMatch JP 8015 'TOYOTA TSUSHO CORPORATION' supported>, <CompanyMatch JP 3116 'TOYOTA BOSHOKU CORPORATION' supported>]

In [3]:
match = find_company("FR0010307819")[0]  # an ISIN
match.market, match.identifier, match.name, match.company().filings(form="annual").latest()

('FR', '969500XXRPGD7HCAFA90', 'LEGRAND', <Filing 370:623156_20260408 300005 ANNUAL filed=2026-04-08 FR/france_oam>)

In [4]:
for market in ("US", "UK", "JP", "FR"):
    print(market, provider_info(market).identifiers)

US ('ticker', 'CIK')
UK ('company number', 'exact company name')
JP ('four-digit securities code', 'EDINET code')
FR ('LEI', 'exact source issuer name')


UK: Companies House search, then the latest filed accounts.

In [5]:
candidates = search_companies("Greggs", market="UK", limit=3)
[(entity.entity_id, entity.name) for entity in candidates]

[('00502851', 'GREGGS PLC'), ('SC702856', 'GREGGS ACADEMIC SERVICES LTD'), ('16546851', 'GREGGS CARGO AND LOGISTICS UK LTD')]

In [6]:
greggs = Company(candidates[0])
accounts = greggs.filings(form="annual").latest()
accounts, accounts.title

(<Filing MzUyMjMwODYxMGFkaXF6a2N4 AA ANNUAL period=2025-12-27 filed=2026-05-19 UK/companies_house>, "Group of companies' accounts made up to 27 December 2025")

France: the regulator's portal by LEI or name. Records keep the source's own codes.

In [7]:
matches = search_companies("Legrand", market="FR", limit=1)
legrand = Company(matches[0])
report = legrand.filings(form="annual").latest()
report, report.title, report.filing_type_raw

(<Filing 370:623156_20260408 300005 ANNUAL filed=2026-04-08 FR/france_oam>, "DOCUMENT D'ENREGISTREMENT UNIVERSEL 2025", '300005')

Japan: EDINET needs an API key (`OPENCORPFILINGS_EDINET_API_KEY`) and finds filings by
scanning its daily lists, so a date window makes a lookup fast.

In [8]:
if os.getenv("OPENCORPFILINGS_EDINET_API_KEY") or os.getenv("EDINET_API_KEY"):
    toyota = Company("7203", market="JP")
    print(toyota.filings(form="annual", start="2026-06-01", end="2026-06-30").latest())
else:
    print("set OPENCORPFILINGS_EDINET_API_KEY to run this cell")

<Filing S100Y8NY 120 ANNUAL period=2026-03-31 filed=2026-06-10 JP/edinet>


A request a route cannot serve fails at once, with the allowed values:

In [9]:
try:
    greggs.filings(form="interim")
except Exception as error:
    print(type(error).__name__, error)

UnsupportedCapabilityError UK/companies_house cannot select form='interim'; its categories are: annual, other (exact native source forms are also accepted)


In [10]:
try:
    provider_info("GB")
except Exception as error:
    print(type(error).__name__, error)

UnsupportedMarketError unsupported market 'GB' (did you mean 'UK'?); `ocf markets` or supported_markets() lists the routes


A collection lists several filings; `to_pandas(n)` needs the `pandas` extra and a bound.

In [11]:
apple = Company("AAPL", market="US")
apple.filings(form="quarterly").to_pandas(4)[["filing_id", "filing_type_raw", "filing_date", "period_end"]]

,filing_id,filing_type_raw,filing_date,period_end
0,0000320193-26-000020,10-Q,2026-07-31,2026-06-27
1,0000320193-26-000013,10-Q,2026-05-01,2026-03-28
2,0000320193-26-000006,10-Q,2026-01-30,2025-12-27
3,0000320193-25-000073,10-Q,2025-08-01,2025-06-28
